<a href="https://colab.research.google.com/github/Epidemic2001/PetProyects_Studies/blob/main/01_Aprendizaje_Autosupervisado_MNIST_EXPLICADO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 1 — Aprendizaje autosupervisado con MNIST

## Objetivo general
En este laboratorio aprenderemos, paso a paso, qué es el **aprendizaje autosupervisado (Self-Supervised Learning, SSL)** y cómo se puede usar para aprender representaciones útiles sin depender de etiquetas creadas manualmente.

Trabajaremos con **MNIST**, un conjunto de imágenes de dígitos manuscritos. Aunque MNIST incluye etiquetas del 0 al 9, durante la etapa autosupervisada **no las utilizaremos**.

La tarea que construiremos será:

**imagen con ruido → autoencoder → imagen reconstruida**

El propio dato original actúa como objetivo de entrenamiento. Por eso hablamos de autosupervisión.

### Conceptos que aparecerán
- **Aprendizaje autosupervisado:** forma de aprendizaje en la que la señal objetivo se obtiene automáticamente desde los propios datos.
- **Autoencoder:** red neuronal que intenta comprimir una entrada y luego reconstruirla.
- **Encoder:** parte del autoencoder que transforma la entrada en una representación compacta.
- **Decoder:** parte que reconstruye la entrada a partir de esa representación.
- **Representación latente:** resumen numérico de la información relevante contenida en los datos.
- **Fine-tuning / reutilización:** uso posterior de una representación aprendida para resolver otra tarea.

### Qué deberá comprender el alumno al terminar
1. La diferencia entre aprendizaje supervisado y autosupervisado.
2. Cómo se puede construir automáticamente una tarea de entrenamiento.
3. Qué función cumplen encoder y decoder.
4. Cómo una representación aprendida sin etiquetas puede reutilizarse después en clasificación.

## 1. Importar librerías

Antes de trabajar con los datos necesitamos cargar las bibliotecas que utilizaremos.

### ¿Qué es una librería?
Una **librería** es un conjunto de funciones y herramientas ya programadas que podemos reutilizar.

En este laboratorio usaremos:
- **NumPy:** operaciones numéricas y manejo de arreglos.
- **Matplotlib:** creación de gráficos y visualización de imágenes.
- **TensorFlow:** framework de aprendizaje profundo.
- **Keras:** interfaz de alto nivel incluida en TensorFlow para construir redes neuronales de manera sencilla.
- **Scikit-learn:** métricas de evaluación para la etapa final de clasificación.

### ¿Por qué fijamos semillas aleatorias?
Muchas operaciones de redes neuronales incluyen aleatoriedad. Definir una semilla ayuda a obtener resultados más reproducibles entre ejecuciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)

# Semillas para obtener resultados más reproducibles
np.random.seed(42)
tf.random.set_seed(42)

## 2. Cargar el dataset MNIST

**MNIST** es uno de los datasets clásicos de aprendizaje automático. Contiene imágenes en escala de grises de dígitos escritos a mano.

Cada imagen tiene:
- tamaño **28 × 28 píxeles**,
- un solo canal de color porque está en escala de grises,
- una etiqueta entre 0 y 9.

### Importante para este laboratorio
Aunque cargaremos `y_train` y `y_test`, durante el **preentrenamiento autosupervisado** no utilizaremos esas etiquetas.

Las reservaremos para una etapa posterior en la que comprobaremos si el encoder aprendió características útiles.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Imágenes de entrenamiento:", x_train.shape)
print("Etiquetas de entrenamiento:", y_train.shape)
print("Imágenes de prueba:", x_test.shape)
print("Etiquetas de prueba:", y_test.shape)

## 3. Visualizar algunas imágenes

Siempre es recomendable observar los datos antes de entrenar un modelo.

### ¿Por qué visualizar?
Porque permite comprobar rápidamente:
- si las imágenes se cargaron correctamente,
- qué aspecto tienen los ejemplos,
- si las etiquetas parecen coherentes,
- si existen problemas evidentes en los datos.

En este bloque mostraremos 10 imágenes junto a sus etiquetas. Las etiquetas solo se usan aquí para fines didácticos.

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Etiqueta: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Normalización de los píxeles

Las imágenes de MNIST vienen originalmente con valores de píxel entre **0 y 255**.

La normalización transforma esos valores al rango **0 a 1** dividiendo por 255.

### ¿Por qué normalizar?
Las redes neuronales suelen entrenar de forma más estable cuando las entradas están en rangos pequeños y comparables.

También agregamos una dimensión extra para representar el canal de color:

`28 × 28` → `28 × 28 × 1`

Ese último `1` indica que la imagen tiene un único canal.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Agregamos un canal para que la forma sea (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Nueva forma:", x_train.shape)
print("Mínimo:", x_train.min())
print("Máximo:", x_train.max())

## 5. Crear la tarea autosupervisada

Aquí construiremos la señal de supervisión automáticamente.

Tomaremos cada imagen original y le agregaremos **ruido gaussiano**.

### ¿Qué es ruido gaussiano?
Es una perturbación aleatoria cuyos valores siguen aproximadamente una distribución normal. Se usa para simular errores, ruido de sensores o degradaciones en una señal.

La tarea será:

- **Entrada:** imagen con ruido.
- **Objetivo:** imagen original.

Esto es autosupervisado porque no necesitamos que una persona etiquete cada ejemplo. El objetivo ya está contenido en los propios datos.

In [ ]:
noise_factor = 0.4

x_train_noisy = x_train + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=x_train.shape
)

x_test_noisy = x_test + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=x_test.shape
)

# Dejamos los valores dentro del intervalo [0, 1]
x_train_noisy = np.clip(x_train_noisy, 0.0, 1.0).astype("float32")
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0).astype("float32")

## 6. Comparar imágenes originales e imágenes con ruido

Antes de entrenar, observaremos si el ruido agregado es suficiente para dificultar la imagen sin destruir completamente su contenido.

### Qué debería observar
La forma general del dígito todavía debería ser reconocible, pero con suficientes perturbaciones como para que la reconstrucción represente una tarea de aprendizaje real.

In [ ]:
n = 6

plt.figure(figsize=(12, 4))

for i in range(n):
    # Original
    plt.subplot(2, n, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Con ruido
    plt.subplot(2, n, n + i + 1)
    plt.imshow(x_train_noisy[i].squeeze(), cmap="gray")
    plt.title("Con ruido")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 7. Construir el autoencoder

Un **autoencoder** es una red neuronal diseñada para aprender una representación comprimida de sus datos y luego reconstruirlos.

Tiene dos componentes principales:

### Encoder
Convierte la imagen de entrada en una representación interna más compacta.

`imagen → representación latente`

### Decoder
Usa esa representación para intentar reconstruir la imagen original.

`representación latente → imagen reconstruida`

### Capas que utilizaremos
- **Conv2D:** detecta patrones espaciales como bordes, curvas o texturas.
- **MaxPooling2D:** reduce el tamaño espacial y conserva información importante.
- **UpSampling2D:** aumenta nuevamente la resolución durante la reconstrucción.
- **Sigmoid:** limita la salida al rango 0–1, coherente con nuestras imágenes normalizadas.

### Idea principal
Si el encoder aprende bien, su representación latente debería capturar información útil del dígito y no solo memorizar el ruido.

In [ ]:
# -------------------------
# ENCODER
# -------------------------
encoder_input = keras.Input(shape=(28, 28, 1), name="imagen_entrada")

x = layers.Conv2D(32, 3, activation="relu", padding="same")(encoder_input)
x = layers.MaxPooling2D(2, padding="same")(x)

x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
latent = layers.MaxPooling2D(2, padding="same", name="representacion_latente")(x)

encoder = keras.Model(encoder_input, latent, name="encoder")

# -------------------------
# DECODER
# -------------------------
x = layers.Conv2D(64, 3, activation="relu", padding="same")(latent)
x = layers.UpSampling2D(2)(x)

x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
x = layers.UpSampling2D(2)(x)

decoded = layers.Conv2D(
    1,
    3,
    activation="sigmoid",
    padding="same",
    name="imagen_reconstruida"
)(x)

autoencoder = keras.Model(encoder_input, decoded, name="autoencoder")

autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

autoencoder.summary()

## 8. Entrenamiento autosupervisado

Ahora entrenaremos el autoencoder.

El código utiliza:

`x_train_noisy` como entrada y `x_train` como objetivo.

### ¿Qué significa entrenar?
Entrenar una red neuronal significa ajustar sus pesos internos para reducir una función de error.

### Función de pérdida
Usaremos **binary crossentropy**. Aunque suele asociarse a clasificación binaria, también puede utilizarse para reconstrucción cuando los píxeles están normalizados entre 0 y 1.

### Epoch
Una **época** corresponde a una pasada completa por todos los datos de entrenamiento.

### Batch
Un **batch** es un pequeño grupo de ejemplos procesados antes de actualizar los pesos.

### Punto clave
En esta fase **no utilizamos `y_train`**. El modelo aprende a partir de la estructura de las propias imágenes.

In [ ]:
history_ssl = autoencoder.fit(
    x_train_noisy,
    x_train,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

## 9. Analizar la curva de pérdida

La **loss** mide qué tan diferente es la reconstrucción producida por el modelo respecto de la imagen original.

Mostraremos:
- pérdida de entrenamiento,
- pérdida de validación.

### Qué debería ocurrir
Idealmente ambas deberían disminuir con las épocas. Si la pérdida de entrenamiento baja mucho pero la de validación empeora, podríamos estar frente a **sobreajuste (overfitting)**.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_ssl.history["loss"], label="Entrenamiento")
plt.plot(history_ssl.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Binary Crossentropy")
plt.title("Aprendizaje del autoencoder")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. Reconstruir imágenes nuevas

Ahora aplicaremos el autoencoder a imágenes del conjunto de prueba.

Compararemos tres versiones:
1. imagen con ruido,
2. reconstrucción generada por el modelo,
3. imagen original.

### Qué queremos comprobar
Si el modelo aprendió una representación útil, la reconstrucción debería conservar la forma del dígito y eliminar parte del ruido.

In [ ]:
reconstructed = autoencoder.predict(x_test_noisy[:10], verbose=0)

n = 10
plt.figure(figsize=(15, 6))

for i in range(n):
    # Imagen ruidosa
    plt.subplot(3, n, i + 1)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Ruido")

    # Reconstrucción
    plt.subplot(3, n, n + i + 1)
    plt.imshow(reconstructed[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Reconstruida")

    # Original
    plt.subplot(3, n, 2 * n + i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Original")

plt.suptitle("Entrada ruidosa → Reconstrucción → Original")
plt.tight_layout()
plt.show()

# 11. Reutilizar el encoder para clasificación

Hasta aquí aprendimos sin utilizar etiquetas de clase.

Ahora usaremos el **encoder preentrenado** como extractor de características para una tarea supervisada.

La nueva arquitectura será:

`imagen → encoder preentrenado → capas densas → clase 0–9`

### ¿Qué significa congelar el encoder?
`encoder.trainable = False` indica que sus pesos no se modificarán durante esta etapa.

Esto permite conservar lo aprendido durante el preentrenamiento autosupervisado.

### Capas nuevas
- **Flatten:** convierte las características en un vector.
- **Dense:** capa completamente conectada.
- **Softmax:** transforma la salida en probabilidades para las 10 clases.

In [ ]:
# Congelamos el encoder para conservar las representaciones aprendidas
encoder.trainable = False

classifier_input = keras.Input(shape=(28, 28, 1))

x = encoder(classifier_input, training=False)
x = layers.Flatten()(x)
x = layers.Dense(64, activation="relu")(x)
classifier_output = layers.Dense(10, activation="softmax")(x)

classifier = keras.Model(
    classifier_input,
    classifier_output,
    name="clasificador_con_encoder_ssl"
)

classifier.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

classifier.summary()

## 12. Entrenar el clasificador

En esta fase sí utilizaremos etiquetas, porque ahora resolveremos una tarea de clasificación.

Para hacer más evidente la utilidad del aprendizaje previo, utilizaremos solo una parte de las imágenes etiquetadas.

### Idea importante
El encoder ya llega con conocimiento adquirido sin etiquetas. Solo necesitamos entrenar la parte final que transforma esas características en clases.

In [ ]:
N_LABELLED = 10000

history_classifier = classifier.fit(
    x_train[:N_LABELLED],
    y_train[:N_LABELLED],
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 13. Evaluación final

Evaluaremos el modelo con datos que no fueron utilizados para entrenar el clasificador.

### Accuracy
La **accuracy** es la proporción total de predicciones correctas.

### Classification report
También calcularemos:
- **Precision:** de los ejemplos predichos como una clase, cuántos eran realmente de esa clase.
- **Recall:** de los ejemplos reales de una clase, cuántos fueron detectados correctamente.
- **F1-score:** equilibrio entre precision y recall.

In [ ]:
test_loss, test_accuracy = classifier.evaluate(x_test, y_test, verbose=0)

print(f"Accuracy de prueba: {test_accuracy:.4f}")

In [ ]:
pred_probs = classifier.predict(x_test, verbose=0)
y_pred = np.argmax(pred_probs, axis=1)

print(classification_report(y_test, y_pred))

# 14. Conclusiones del laboratorio

Hemos utilizado dos etapas claramente diferenciadas.

## Etapa A — Aprendizaje autosupervisado
`imagen con ruido → autoencoder → reconstrucción`

En esta etapa el modelo aprendió sin etiquetas de clase.

## Etapa B — Reutilización supervisada
`imagen → encoder aprendido → clasificador → dígito`

Aquí aprovechamos la representación aprendida para resolver una tarea distinta.

## Conceptos clave
- **Self-Supervised Learning:** aprende usando objetivos generados desde los propios datos.
- **Autoencoder:** aprende a comprimir y reconstruir información.
- **Encoder:** obtiene características relevantes.
- **Decoder:** reconstruye la entrada.
- **Latent representation:** representación numérica interna de los datos.
- **Feature extraction:** uso de un modelo para producir características útiles.

## Glosario rápido
| Concepto | Descripción breve |
|---|---|
| Tensor | arreglo numérico multidimensional |
| Feature | característica que ayuda a describir un dato |
| Epoch | pasada completa por el conjunto de entrenamiento |
| Batch | subconjunto procesado en una actualización |
| Loss | medida del error del modelo |
| Overfitting | cuando el modelo aprende demasiado el entrenamiento y generaliza peor |
| Encoder | parte que comprime o extrae características |
| Decoder | parte que reconstruye la entrada |
| Representación latente | resumen interno aprendido por la red |

## Preguntas para discusión
1. ¿Por qué este laboratorio es autosupervisado y no supervisado tradicional?
2. ¿De dónde proviene el objetivo del autoencoder?
3. ¿Qué información esperamos que aprenda el encoder?
4. ¿Qué ocurriría si el ruido fuera demasiado alto?
5. ¿Por qué puede ser útil el SSL cuando existen pocos datos etiquetados?